# Final Project: Mini Campus Delivery Robot

## Project Team
* **Tleubergenov Bizhan:** Code Implementation & Algorithm Development
* **Rakhimzhanova Ashanti:** Scenario Design, Project Descriptions, and Documentation

---

## Project Overview
**Scenario:** A delivery robot carries a package across a small campus map.
This project demonstrates the integration of three core AI algorithms from the course into a single real-world scenario. It does not require trained ML models; instead, it uses predefined probabilities, transition tables, and a toy map to clearly showcase the logic of each method.

---

## 1. Search Algorithm: A* (A-star)
**Goal:** Find the optimal route on a 5×5 grid with walls.
* **Input:** A 5x5 grid, a starting point at `(0,0)`, a goal at `(4,4)`, and obstacles located at `(1,1)` and `(2,2)`.
* **Key Calculation:** The algorithm explores the grid using a priority queue based on the cost function `f(n) = g(n) + h(n)`, where `g(n)` is the step cost and `h(n)` is the Manhattan distance heuristic to the goal.
* **Output:** A list of coordinates representing the shortest path, e.g., `[(0, 0), (1, 0), (2, 0)...]`.
* **Explanation:** A* efficiently calculates the shortest route around obstacles by prioritizing paths that appear closer to the destination.

---

## 2. Probabilistic Reasoning: Bayesian Inference
**Goal:** Update the probability of an obstacle blocking the path after receiving a noisy sensor signal.
* **Input:** A prior probability of an obstacle (30%), a True Positive sensor rate (90%), a False Positive rate (10%), and one active sensor alert.
* **Key Calculation:** Applies Bayes' Theorem to calculate the posterior probability: `P(Obstacle | Alert) = [P(Alert | Obstacle) * P(Obstacle)] / P(Alert)`.
* **Output:** The updated (posterior) probability of the obstacle being present (approx. 77%).
* **Explanation:** The algorithm mathematically corrects our initial belief about the environment by weighing the reliability of the new sensor reading.

---

## 3. Sequential Decision Making: Markov Decision Process (MDP)
**Goal:** Choose the best immediate action (*Move* or *Wait*) in an uncertain corridor.
* **Input:** 2 environmental states (*Clear* or *Blocked*) using the probabilities derived from the Bayesian step, and a predefined reward table (e.g., *Move* while *Blocked* = -15 penalty).
* **Key Calculation:** Computes the expected reward (one Bellman update) for each action by multiplying the state probabilities by their respective rewards and summing them up.
* **Output:** The recommended action with the highest expected value (e.g., *Wait*).
* **Explanation:** MDP evaluates the mathematical risks and rewards under uncertainty to make the safest and most advantageous decision for the robot.

In [ ]:
import heapq

def main():
    print("=== 1. Поиск пути: Алгоритм A* ===")
    # Входные данные: Сетка 5x5, 2 стены
    grid_size = 5
    start = (0, 0)
    goal = (4, 4)
    walls = {(1, 1), (2, 2)}

    # Ключевые вычисления (Key steps)
    frontier = []
    heapq.heappush(frontier, (0, start))
    came_from = {start: None}
    cost_so_far = {start: 0}

    while frontier:
        _, current = heapq.heappop(frontier)
        if current == goal:
            break

        for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            next_node = (current[0] + dx, current[1] + dy)
            # Проверка границ и стен
            if 0 <= next_node[0] < grid_size and 0 <= next_node[1] < grid_size and next_node not in walls:
                new_cost = cost_so_far[current] + 1
                if next_node not in cost_so_far or new_cost < cost_so_far[next_node]:
                    cost_so_far[next_node] = new_cost
                    # Эвристика: Манхэттенское расстояние
                    priority = new_cost + abs(goal[0] - next_node[0]) + abs(goal[1] - next_node[1])
                    heapq.heappush(frontier, (priority, next_node))
                    came_from[next_node] = current

    path = []
    curr = goal
    while curr != start:
        path.append(curr)
        curr = came_from[curr]
    path.append(start)
    path.reverse()

    print(f"Вход: Сетка {grid_size}x{grid_size}, Старт: {start}, Цель: {goal}, Стены: {walls}")
    print(f"Выход: Найденный маршрут -> {path}\n")

    print("=== 2. Вероятностный вывод: Теорема Байеса ===")
    # Входные данные: 1 наблюдение сенсора
    prior_obstacle = 0.30 # Изначальная вероятность препятствия (30%)
    prob_alert_given_obstacle = 0.90 # True positive (сенсор прав)
    prob_alert_given_clear = 0.10 # False positive (сенсор ошибся)

    # Ключевые вычисления: P(Obstacle | Alert)
    prob_alert = (prob_alert_given_obstacle * prior_obstacle) + (prob_alert_given_clear * (1 - prior_obstacle))
    posterior_obstacle = (prob_alert_given_obstacle * prior_obstacle) / prob_alert

    print("Вход: Априорная вероятность = 0.3, Сенсор обнаружил преграду")
    print(f"Выход: Обновленная вероятность препятствия -> {posterior_obstacle:.2f}\n")

    print("=== 3. Последовательные решения: MDP (один шаг) ===")
    # Входные данные: 2 состояния, 2 действия
    prob_blocked = posterior_obstacle
    prob_clear = 1.0 - posterior_obstacle

    # Таблица наград (Rewards):
    # Действие Move: Свободно = +10, Заблокировано = -15 (авария)
    # Действие Wait: Свободно = -2 (потеря времени), Заблокировано = +5 (безопасное ожидание)

    # Ключевые вычисления: Ожидаемая выгода
    expected_move = (prob_clear * 10) + (prob_blocked * -15)
    expected_wait = (prob_clear * -2) + (prob_blocked * 5)

    best_action = "Move" if expected_move > expected_wait else "Wait"

    print(f"Вход: Награда Move = {expected_move:.2f}, Награда Wait = {expected_wait:.2f}")
    print(f"Выход: Рекомендуемое действие робота -> {best_action}")

if __name__ == "__main__":
    main()

=== 1. Поиск пути: Алгоритм A* ===
Вход: Сетка 5x5, Старт: (0, 0), Цель: (4, 4), Стены: {(1, 1), (2, 2)}
Выход: Найденный маршрут -> [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4)]

=== 2. Вероятностный вывод: Теорема Байеса ===
Вход: Априорная вероятность = 0.3, Сенсор обнаружил преграду
Выход: Обновленная вероятность препятствия -> 0.79

=== 3. Последовательные решения: MDP (один шаг) ===
Вход: Награда Move = -9.85, Награда Wait = 3.56
Выход: Рекомендуемое действие робота -> Wait
